# Notebook 3: VRDFormer Model Architecture Deep Dive

**Goal:** Build the VRDFormer model, trace tensor shapes through every layer, understand the tracking mechanism, and count parameters.

VRDFormer is a **DETR-based video relation detection model** with a two-stage design:
- **Stage 1:** Subject-object pair detection with tracking (Hungarian matching + recurrent queries)
- **Stage 2:** Relation classification over time (ROI-initialized queries + temporal memory)

The architecture follows: **ResNet Backbone → Transformer Encoder → Transformer Decoder → Prediction Heads** (duplicated for subject AND object).

## 1. Setup & Build the Model

In [1]:
import os
import sys
import json
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from argparse import Namespace

# Add repo root
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

# Monkey-patch import blocks for notebook environment
import util.misc as utils

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

PyTorch: 2.13.0+cpu
CUDA available: False


In [2]:
# Build args mimicking configs/vidvrd_stage1.json
def build_stage1_args():
    args = Namespace()
    args.dataset = 'vidvrd'
    args.stage = 1
    args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Backbone
    args.backbone = 'resnet101'
    args.lr_backbone = 1e-5
    args.dilation = False
    args.position_embedding = 'sine'
    
    # Transformer
    args.enc_layers = 6
    args.dec_layers = 6
    args.hidden_dim = 256
    args.dim_feedforward = 2048
    args.nheads = 8
    args.dropout = 0.1
    args.num_queries = 100
    args.pre_norm = False
    args.aux_loss = True
    
    # Deformable (off for vanilla)
    args.deformable = False
    args.num_feature_levels = 1
    args.with_box_refine = False
    args.overflow_boxes = False
    args.dec_n_points = 4
    args.enc_n_points = 4
    
    # Multi-frame
    args.multi_frame_attention = False
    args.multi_frame_encoding = False
    args.merge_frame_features = False
    args.multi_frame_attention_separate_encoder = False
    
    # Loss
    args.focal_loss = True
    args.focal_alpha = 0.25
    args.focal_gamma = 2
    args.obj_loss_coef = 1
    args.verb_loss_coef = 1
    args.bbox_loss_coef = 5
    args.giou_loss_coef = 2
    args.eos_coef = 0.1
    
    # Matcher
    args.set_cost_class = 1
    args.set_cost_bbox = 5
    args.set_cost_giou = 2
    args.set_cost_sub_class = 0.5
    args.set_cost_obj_class = 0.5
    args.set_cost_verb_class = 1
    
    # Tracking
    args.tracking = True
    args.track_attention = False
    args.track_query_false_positive_prob = 0.1
    args.track_query_false_negative_prob = 0.4
    args.track_query_false_positive_eos_weight = True
    args.track_backprop_prev_frame = False
    
    # Data paths (not needed for model building)
    args.vidvrd_path = ''
    args.vidor_path = ''
    args.coco_path = ''
    
    args.distributed = False
    args.gpu = 0
    
    return args

args = build_stage1_args()
print('Args built. Key settings:')
print(f'  backbone={args.backbone}, hidden_dim={args.hidden_dim}')
print(f'  enc_layers={args.enc_layers}, dec_layers={args.dec_layers}')
print(f'  nheads={args.nheads}, dim_feedforward={args.dim_feedforward}')
print(f'  num_queries={args.num_queries}, tracking={args.tracking}')

Args built. Key settings:
  backbone=resnet101, hidden_dim=256
  enc_layers=6, dec_layers=6
  nheads=8, dim_feedforward=2048
  num_queries=100, tracking=True


In [3]:
# Build the model
from models import build_model

print('Building VRDFormer...')
model, criterion, weight_dict = build_model(args)
model.to(args.device)

print(f'\nModel class: {type(model).__name__}')
print(f'Criterion class: {type(criterion).__name__}')
print(f'\nWeight dict:')
for k, v in weight_dict.items():
    print(f'  {k}: {v}')

Building VRDFormer...


/mnt/c/Users/Samuel Oliveira/Desktop/CS/VRDFormer_VRD/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/c/Users/Samuel Oliveira/Desktop/CS/VRDFormer_VRD/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Model class: VRDFormerTracking
Criterion class: SetCriterionTrack

Weight dict:
  loss_ce: 1
  loss_ce_verb: 1
  loss_bbox: 5
  loss_giou: 2
  loss_ce_0: 1
  loss_ce_verb_0: 1
  loss_bbox_0: 5
  loss_giou_0: 2
  loss_ce_1: 1
  loss_ce_verb_1: 1
  loss_bbox_1: 5
  loss_giou_1: 2
  loss_ce_2: 1
  loss_ce_verb_2: 1
  loss_bbox_2: 5
  loss_giou_2: 2
  loss_ce_3: 1
  loss_ce_verb_3: 1
  loss_bbox_3: 5
  loss_giou_3: 2
  loss_ce_4: 1
  loss_ce_verb_4: 1
  loss_bbox_4: 5
  loss_giou_4: 2
  loss_ce_enc: 1
  loss_ce_verb_enc: 1
  loss_bbox_enc: 5
  loss_giou_enc: 2


## 2. Model Architecture Overview

```
VRDFormerTracking (MRO: TrackingBase, VRDFormer)
  |
  ├── backbone: Joiner(Backbone(ResNet101), PositionEmbeddingSine)
  |     input:  (B, 3, H, W)
  |     output: (B, 2048, H/32, W/32) features + (B, 256, H/32, W/32) pos encoding
  |
  ├── input_proj: Conv2d(2048 -> 256, kernel=1) + GroupNorm
  |     input:  (B, 2048, H/32, W/32)
  |     output: (B, 256, H/32, W/32)
  |
  ├── query_embed: nn.Embedding(num_queries, hidden_dim*2)
  |     input:  index [0..num_queries-1]
  |     output: (num_queries, 512) -> split into query_pos(256) + tgt(256)
  |
  ├── transformer: Transformer
  |     encoder: 6x TransformerEncoderLayer (self-attn + FFN)
  |     decoder: 6x TransformerDecoderLayer (self-attn + cross-attn + FFN)
  |     input:  features [seq_len, B, 256], query [num_q, B, 256]
  |     output: hs [6, B, num_q, 256], memory [B, seq_len, 256]
  |
  └── prediction heads (applied per decoder layer):
       sub_class_embed:  Linear(256 -> 36)      [35 objects + 1 no-object]
       obj_class_embed:  Linear(256 -> 36)
       verb_class_embed: Linear(256 -> 132)      [multi-label, no no-verb class]
       sub_bbox_embed:   MLP(256 -> 256 -> 4)    [cxcywh, sigmoid]
       obj_bbox_embed:   MLP(256 -> 256 -> 4)
```

**Key design:** Every prediction head is **duplicated** for subject and object. This is the core insight of VRDFormer — each query predicts a **pair** of bounding boxes and classes, not a single object.

## 3. Count Parameters

Let's count trainable parameters broken down by component.

In [4]:
def count_params(module, name='total'):
    # Handle nn.Parameter (raw tensor) vs nn.Module
    if isinstance(module, torch.nn.Parameter):
        return module.numel() if module.requires_grad else 0
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

def count_all_params(module, name='total'):
    if isinstance(module, torch.nn.Parameter):
        return module.numel()
    return sum(p.numel() for p in module.parameters())

print('=== PARAMETER COUNTS ===')
print(f'{"Component":<35} {"Trainable":>12} {"Total":>12}')
print('-' * 60)

# Backbone
n_backbone = count_params(model.backbone)
n_backbone_total = count_all_params(model.backbone)
print(f'{"Backbone (ResNet101+FrozenBN)":<35} {n_backbone/1e6:>9.2f}M {n_backbone_total/1e6:>9.2f}M')

# Transformer encoder
n_enc = count_params(model.transformer.encoder)
print(f'{"Transformer Encoder":<35} {n_enc/1e6:>9.2f}M {n_enc/1e6:>9.2f}M')

# Transformer decoder
n_dec = count_params(model.transformer.decoder)
print(f'{"Transformer Decoder":<35} {n_dec/1e6:>9.2f}M {n_dec/1e6:>9.2f}M')

# Level embed (raw Parameter, not a Module)
n_level = count_params(model.transformer.level_embed)
print(f'{"Level Embedding":<35} {n_level/1e6:>9.2f}M {n_level/1e6:>9.2f}M')

# Query embeddings
n_query = count_params(model.query_embed)
print(f'{"Query Embeddings":<35} {n_query/1e6:>9.2f}M {n_query/1e6:>9.2f}M')

# Input projection
n_proj = count_params(model.input_proj)
print(f'{"Input Projection":<35} {n_proj/1e6:>9.2f}M {n_proj/1e6:>9.2f}M')

# Prediction heads
n_sub_cls = count_params(model.sub_class_embed)
n_obj_cls = count_params(model.obj_class_embed)
n_verb_cls = count_params(model.verb_class_embed)
n_sub_box = count_params(model.sub_bbox_embed)
n_obj_box = count_params(model.obj_bbox_embed)
n_heads = n_sub_cls + n_obj_cls + n_verb_cls + n_sub_box + n_obj_box
print(f'{"Prediction Heads (sub cls)":<35} {n_sub_cls/1e6:>9.2f}M {n_sub_cls/1e6:>9.2f}M')
print(f'{"Prediction Heads (obj cls)":<35} {n_obj_cls/1e6:>9.2f}M {n_obj_cls/1e6:>9.2f}M')
print(f'{"Prediction Heads (verb cls)":<35} {n_verb_cls/1e6:>9.2f}M {n_verb_cls/1e6:>9.2f}M')
print(f'{"Prediction Heads (sub box)":<35} {n_sub_box/1e6:>9.2f}M {n_sub_box/1e6:>9.2f}M')
print(f'{"Prediction Heads (obj box)":<35} {n_obj_box/1e6:>9.2f}M {n_obj_box/1e6:>9.2f}M')

print('-' * 60)
n_total = count_params(model)
n_total_all = count_all_params(model)
print(f'{"TOTAL":<35} {n_total/1e6:>9.2f}M {n_total_all/1e6:>9.2f}M')

# Per-layer breakdown
print()
print('=== PER-LAYER DETAILS ===')
print(f'Encoder: {args.enc_layers} layers, each = 1 self-attn ({args.hidden_dim}d, {args.nheads}h) + FFN ({args.hidden_dim}->{args.dim_feedforward})')
print(f'Decoder: {args.dec_layers} layers, each = 1 self-attn + 1 cross-attn + FFN')
print(f'  Self-attention:  {args.hidden_dim}d, {args.nheads} heads, dropout={args.dropout}')
print(f'  Cross-attention: {args.hidden_dim}d, {args.nheads} heads')
print(f'  FFN:             {args.hidden_dim}->{args.dim_feedforward}->{args.hidden_dim}')
print(f'Num queries: {args.num_queries} pairs (each query predicts 1 subject + 1 object + 1 relation)')


=== PARAMETER COUNTS ===
Component                              Trainable        Total
------------------------------------------------------------
Backbone (ResNet101+FrozenBN)           42.17M     42.39M
Transformer Encoder                      7.89M      7.89M
Transformer Decoder                      9.47M      9.47M
Level Embedding                          0.00M      0.00M
Query Embeddings                         0.05M      0.05M
Input Projection                         0.53M      0.53M
Prediction Heads (sub cls)               0.01M      0.01M
Prediction Heads (obj cls)               0.01M      0.01M
Prediction Heads (verb cls)              0.03M      0.03M
Prediction Heads (sub box)               0.13M      0.13M
Prediction Heads (obj box)               0.13M      0.13M
------------------------------------------------------------
TOTAL                                   60.43M     60.65M

=== PER-LAYER DETAILS ===
Encoder: 6 layers, each = 1 self-attn (256d, 8h) + FFN (256->2048)
D

In [5]:
# Create a dummy input image
H, W = 480, 640  # typical image size
dummy_img = torch.randn(1, 3, H, W)
dummy_mask = torch.zeros(1, H, W, dtype=torch.bool)
dummy_nt = utils.NestedTensor(dummy_img, dummy_mask)

print(f'Input: {tuple(dummy_img.shape)} -> NestedTensor')

# Backbone forward
with torch.no_grad():
    features, pos = model.backbone(dummy_nt)

print(f'\n=== BACKBONE OUTPUT ===')
print(f'Number of feature levels: {len(features)}')
print(f'(With num_feature_levels={args.num_feature_levels}, only layer4 is returned)')
for i, feat in enumerate(features):
    print(f'Level {i} (stride {model.backbone.strides[i] if i < len(model.backbone.strides) else "?"}): '
          f'features={tuple(feat.tensors.shape)}, pos={tuple(pos[i].shape)}')

print(f'\nNum channels: {model.backbone.num_channels}')
print(f'Strides: {model.backbone.strides}')

# VRDFormer uses features[-3:] — but with num_feature_levels=1, this is just [layer4]
features_used = features[-3:]
print(f'\nFeatures used by VRDFormer (features[-3:]): {len(features_used)} levels')
for i, feat in enumerate(features_used):
    print(f'  Level {i}: {tuple(feat.tensors.shape)}')


Input: (1, 3, 480, 640) -> NestedTensor



=== BACKBONE OUTPUT ===
Number of feature levels: 1
(With num_feature_levels=1, only layer4 is returned)
Level 0 (stride 32): features=(1, 2048, 15, 20), pos=(1, 256, 15, 20)

Num channels: [2048]
Strides: [32]

Features used by VRDFormer (features[-3:]): 1 levels
  Level 0: (1, 2048, 15, 20)


In [6]:
# Show what input_proj does
print('=== INPUT PROJECTION ===')
src = features_used[0]  # NestedTensor
print(f'Input: {tuple(src.tensors.shape)} (2048 channels)')

with torch.no_grad():
    projected = model.input_proj[0](src.tensors)
print(f'After input_proj[0] (1x1 Conv): {tuple(projected.shape)} (256 channels)')


=== INPUT PROJECTION ===
Input: (1, 2048, 15, 20) (2048 channels)
After input_proj[0] (1x1 Conv): (1, 256, 15, 20) (256 channels)


In [7]:
# Manually trace through the transformer (mimicking VRDFormer.forward)
print('=== TRANSFORMER FORWARD TRACE ===')

# Step 1: Build src_list, mask_list, pos_list (as VRDFormer.forward does)
# With num_feature_levels=1, there is only 1 feature level per frame
src_list = []
mask_list = []
pos_list = []

for l, feat in enumerate(features_used):
    src, mask = feat.decompose()
    projected_src = model.input_proj[0](src)  # 1x1 conv: 2048 -> 256
    src_list.append(projected_src)
    mask_list.append(mask)
    pos_list.append(pos[l])  # same index since features_used == features[-3:] aligns with pos[-3:]

print(f'src_list: {len(src_list)} levels -> shapes: {[tuple(s.shape) for s in src_list]}')
print(f'mask_list: {len(mask_list)} levels -> shapes: {[tuple(m.shape) for m in mask_list]}')
print(f'pos_list:  {len(pos_list)} levels -> shapes: {[tuple(p.shape) for p in pos_list]}')


=== TRANSFORMER FORWARD TRACE ===
src_list: 1 levels -> shapes: [(1, 256, 15, 20)]
mask_list: 1 levels -> shapes: [(1, 15, 20)]
pos_list:  1 levels -> shapes: [(1, 256, 15, 20)]


In [8]:
# Step 2: Flatten features for transformer
src_flatten = []
mask_flatten = []
lvl_pos_embed_flatten = []

for lvl, (src, mask_obj, pos_embed) in enumerate(zip(src_list, mask_list, pos_list)):
    bs, c, h, w = src.shape
    src_f = src.flatten(2).transpose(1, 2)  # [B, C, H, W] -> [B, H*W, C]
    mask_f = mask_obj.flatten(1)             # [B, H, W] -> [B, H*W]
    pos_f = pos_embed.flatten(2).transpose(1, 2)  # [B, C, H, W] -> [B, H*W, C]
    
    lvl_pos = pos_f + model.transformer.level_embed[lvl].view(1, 1, -1)
    lvl_pos_embed_flatten.append(lvl_pos)
    src_flatten.append(src_f)
    mask_flatten.append(mask_f)
    
    print(f'Level {lvl}: src [{bs}, {c}, {h}, {w}] -> flattened [{bs}, {h*w}, {c}]')

# Concatenate all levels
src_flatten = torch.cat(src_flatten, 1)           # [B, total_seq, C]
mask_flatten = torch.cat(mask_flatten, 1)          # [B, total_seq]
lvl_pos_embed_flatten = torch.cat(lvl_pos_embed_flatten, 1)  # [B, total_seq, C]

total_seq = src_flatten.shape[1]
print(f'\nAfter concat: src={tuple(src_flatten.shape)}, '
      f'mask={tuple(mask_flatten.shape)}, '
      f'pos_embed={tuple(lvl_pos_embed_flatten.shape)}')
print(f'total_seq length: {total_seq} (sum of H_l*W_l)')

Level 0: src [1, 256, 15, 20] -> flattened [1, 300, 256]

After concat: src=(1, 300, 256), mask=(1, 300), pos_embed=(1, 300, 256)
total_seq length: 300 (sum of H_l*W_l)


In [9]:
# Step 3: Encoder forward
src_for_enc = src_flatten.transpose(0, 1)  # [total_seq, B, C]
pos_for_enc = lvl_pos_embed_flatten.transpose(0, 1)

print(f'Encoder input: src={tuple(src_for_enc.shape)}, pos={tuple(pos_for_enc.shape)}')

with torch.no_grad():
    memory = model.transformer.encoder(
        src_for_enc,
        pos=pos_for_enc,
        src_key_padding_mask=mask_flatten
    )

print(f'Encoder output (memory): {tuple(memory.shape)}')
print(f'  shape = (total_seq, B, C) = same dimensions as input')

Encoder input: src=(300, 1, 256), pos=(300, 1, 256)


Encoder output (memory): (300, 1, 256)
  shape = (total_seq, B, C) = same dimensions as input


In [10]:
# Step 4: Decoder forward
query_embed = model.query_embed.weight  # [num_queries, 512]
print(f'Query embedding weight: {tuple(query_embed.shape)}')

# Split into query_pos and tgt
c = model.hidden_dim  # 256
query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)  # [num_q, B, 512]
query_pos, tgt = torch.split(query_embed, c, dim=2)  # each [num_q, B, 256]
tgt = torch.zeros_like(query_pos)  # zero-initialize content

print(f'query_pos: {tuple(query_pos.shape)} (split from query_embed weight)')
print(f'tgt:       {tuple(tgt.shape)} (zero-initialized)')
print(f'memory:    {tuple(memory.shape)} (from encoder)')

with torch.no_grad():
    hs = model.transformer.decoder(
        tgt,
        memory,
        memory_key_padding_mask=mask_flatten,
        pos=pos_for_enc,
        query_pos=query_pos,
    )

print(f'\nDecoder output (hs): {tuple(hs.shape)}')
print(f'  dim 0: num_decoder_layers ({args.dec_layers})')
print(f'  dim 1: num_queries ({args.num_queries})')
print(f'  dim 2: batch_size ({bs})')
print(f'  dim 3: hidden_dim ({args.hidden_dim})')

Query embedding weight: (100, 512)
query_pos: (100, 1, 256) (split from query_embed weight)
tgt:       (100, 1, 256) (zero-initialized)
memory:    (300, 1, 256) (from encoder)

Decoder output (hs): (6, 100, 1, 256)
  dim 0: num_decoder_layers (6)
  dim 1: num_queries (100)
  dim 2: batch_size (1)
  dim 3: hidden_dim (256)


## 6. Trace the Prediction Heads

Each decoder layer output goes through 5 prediction heads. All results from the last layer form the main output; intermediate layers form auxiliary outputs.

In [11]:
# Apply prediction heads to the last decoder layer output
hs_last = hs[-1]  # [num_q, B, 256]
hs_last = hs_last.transpose(0, 1)  # [B, num_q, 256]

print('=== PREDICTION HEADS (Stage 1) ===')
print(f'Input: hs_last = {tuple(hs_last.shape)}')
print()

with torch.no_grad():
    # Subject classification
    sub_logits = model.sub_class_embed[0](hs_last)
    print(f'sub_class_embed:  {tuple(hs_last.shape)} -> {tuple(sub_logits.shape)}')
    print(f'  Interpreted as: {args.num_queries} queries x {sub_logits.shape[-1]} classes (35 obj + 1 no-obj)')
    
    # Object classification
    obj_logits = model.obj_class_embed[0](hs_last)
    print(f'obj_class_embed:  {tuple(hs_last.shape)} -> {tuple(obj_logits.shape)}')
    
    # Verb classification (multi-label)
    verb_logits = model.verb_class_embed[0](hs_last)
    print(f'verb_class_embed: {tuple(hs_last.shape)} -> {tuple(verb_logits.shape)}')
    print(f'  Interpreted as: {args.num_queries} queries x {verb_logits.shape[-1]} verbs (multi-label sigmoid)')
    
    # Subject bounding box
    sub_boxes = model.sub_bbox_embed[0](hs_last)
    sub_boxes = sub_boxes.sigmoid()
    print(f'sub_bbox_embed:   {tuple(hs_last.shape)} -> {tuple(sub_boxes.shape)} (cx,cy,w,h in [0,1])')
    
    # Object bounding box
    obj_boxes = model.obj_bbox_embed[0](hs_last)
    obj_boxes = obj_boxes.sigmoid()
    print(f'obj_bbox_embed:   {tuple(hs_last.shape)} -> {tuple(obj_boxes.shape)} (cx,cy,w,h in [0,1])')

=== PREDICTION HEADS (Stage 1) ===
Input: hs_last = (1, 100, 256)

sub_class_embed:  (1, 100, 256) -> (1, 100, 35)
  Interpreted as: 100 queries x 35 classes (35 obj + 1 no-obj)
obj_class_embed:  (1, 100, 256) -> (1, 100, 35)
verb_class_embed: (1, 100, 256) -> (1, 100, 132)
  Interpreted as: 100 queries x 132 verbs (multi-label sigmoid)
sub_bbox_embed:   (1, 100, 256) -> (1, 100, 4) (cx,cy,w,h in [0,1])
obj_bbox_embed:   (1, 100, 256) -> (1, 100, 4) (cx,cy,w,h in [0,1])


In [12]:
# Show the MLP structure for bbox_embed
print('=== MLP Structure (sub_bbox_embed) ===')
for i, layer in enumerate(model.sub_bbox_embed[0].layers):
    w = layer.weight
    b = layer.bias
    act = 'ReLU' if i < len(model.sub_bbox_embed[0].layers) - 1 else 'Identity'
    print(f'  Layer {i}: Linear({w.shape[1]}, {w.shape[0]}) + {act}')

=== MLP Structure (sub_bbox_embed) ===
  Layer 0: Linear(256, 256) + ReLU
  Layer 1: Linear(256, 256) + ReLU
  Layer 2: Linear(256, 4) + Identity


## 7. Understanding the Tracking Query Mechanism

This is the most innovative part of VRDFormer Stage 1. Tracking queries are **recurrent** — the previous frame's decoder hidden states become the current frame's query content.

In [13]:
print('=== TRACKING QUERY MECHANISM ===')
print()
print('How it works:')
print('  1. Previous frame is run through the model (no_grad)')
print('  2. Hungarian matcher links prev predictions to prev targets via track IDs')
print('  3. Matched queries hs_embed are extracted from prev_out')
print('  4. These are written to current targets as track_query_hs_embeds')
print('  5. In the transformer decoder:')
print('     - Track queries: query_pos = ZEROS, tgt = prev_hs_embed (RECYCLED)')
print('     - Static queries: query_pos = LEARNED, tgt = zeros (FRESH)')
print('  6. Decoder input = [track_query_0, ..., track_query_M, static_query_0, ..., static_query_99]')
print()
print('Why this works:')
print('  - Track queries carry content (hs_embed) from previous frame -> temporal consistency')
print('  - Zero query_pos means track queries attend purely by content similarity (no spatial bias)')
print('  - Static queries use learned positional priors (spatial anchor points)')
print('  - Track queries are "forced matched" to their known target indices (cost=-1)')
print()
print('Visualization:')
print('  Frame t-1:  query_i -> predicts (sub_box, obj_box, verb)')
print('  Frame t:    query_i (same hs_embed!) -> predicts updated (sub_box, obj_box, verb)')
print('  The query_i identity persists across frames via track_query_hs_embeds')

=== TRACKING QUERY MECHANISM ===

How it works:
  1. Previous frame is run through the model (no_grad)
  2. Hungarian matcher links prev predictions to prev targets via track IDs
  3. Matched queries hs_embed are extracted from prev_out
  4. These are written to current targets as track_query_hs_embeds
  5. In the transformer decoder:
     - Track queries: query_pos = ZEROS, tgt = prev_hs_embed (RECYCLED)
     - Static queries: query_pos = LEARNED, tgt = zeros (FRESH)
  6. Decoder input = [track_query_0, ..., track_query_M, static_query_0, ..., static_query_99]

Why this works:
  - Track queries carry content (hs_embed) from previous frame -> temporal consistency
  - Zero query_pos means track queries attend purely by content similarity (no spatial bias)
  - Static queries use learned positional priors (spatial anchor points)
  - Track queries are "forced matched" to their known target indices (cost=-1)

Visualization:
  Frame t-1:  query_i -> predicts (sub_box, obj_box, verb)
  Frame 

In [14]:
# Show the tracking-related fields that VRDFormerTracking adds to targets
print('=== Tracking Fields in Targets ===')
print('After add_track_queries_to_targets(), targets gain:')
print('  target["track_query_hs_embeds"]    -> (M, hidden_dim)  prev decoder states')
print('  target["track_query_sub_boxes"]     -> (M, 4)  prev predicted sub boxes')
print('  target["track_query_obj_boxes"]     -> (M, 4)  prev predicted obj boxes')
print('  target["track_query_match_ids"]     -> (M,)  indices into current target boxes')
print('  target["track_queries_mask"]        -> (M+num_q,)  True for all track queries')
print('  target["track_queries_fal_pos_mask"] -> (M+num_q,)  True only for false positives')
print()
print('  where M = number of matched queries from prev frame (varies per sample)')
print()
print('In the decoder (Transformer.forward):')
print('  prev_hs_embed = stack([t["track_query_hs_embeds"] for t in targets])')
print('  prev_hs_embed = prev_hs_embed.transpose(0, 1)  # [M_max, B, 256]')
print('  prev_query_embed = zeros_like(prev_hs_embed)     # zero position')
print('  query_embed = cat([prev_query_embed, query_embed])  # [M_max+num_q, B, 256]')
print('  tgt = cat([prev_hs_embed, tgt])                    # [M_max+num_q, B, 256]')

=== Tracking Fields in Targets ===
After add_track_queries_to_targets(), targets gain:
  target["track_query_hs_embeds"]    -> (M, hidden_dim)  prev decoder states
  target["track_query_sub_boxes"]     -> (M, 4)  prev predicted sub boxes
  target["track_query_obj_boxes"]     -> (M, 4)  prev predicted obj boxes
  target["track_query_match_ids"]     -> (M,)  indices into current target boxes
  target["track_queries_mask"]        -> (M+num_q,)  True for all track queries
  target["track_queries_fal_pos_mask"] -> (M+num_q,)  True only for false positives

  where M = number of matched queries from prev frame (varies per sample)

In the decoder (Transformer.forward):
  prev_hs_embed = stack([t["track_query_hs_embeds"] for t in targets])
  prev_hs_embed = prev_hs_embed.transpose(0, 1)  # [M_max, B, 256]
  prev_query_embed = zeros_like(prev_hs_embed)     # zero position
  query_embed = cat([prev_query_embed, query_embed])  # [M_max+num_q, B, 256]
  tgt = cat([prev_hs_embed, tgt])             

## 8. Stage 2 Architecture (Relation Classification)

Stage 2 is fundamentally different from Stage 1:
- **No learned queries** — queries are initialized from ground-truth boxes via ROI Align
- **No Hungarian matching** — track IDs provide direct correspondence
- **No box loss** — only classification losses
- **Temporal pooling** — embeddings accumulated per tracklet, mean-pooled at end of clip

In [15]:
# Build Stage 2 model for comparison
args_s2 = Namespace(**{k:v for k,v in vars(args).items()})
args_s2.stage = 2
args_s2.tracking = False
args_s2.multi_frame_attention = False

model_s2, criterion_s2, weight_dict_s2 = build_model(args_s2)
model_s2.to(args.device)

print(f'Stage 2 model: {type(model_s2).__name__}')
print(f'Criterion: {type(criterion_s2).__name__}')
print(f'Weight dict: {weight_dict_s2}')
print()

# Check for stage-2-specific modules
print('=== STAGE 2 SPECIFIC MODULES ===')
if hasattr(model_s2.transformer, 'so_linear'):
    print(f'so_linear: {model_s2.transformer.so_linear}')
    print(f'  Purpose: Fuses subject + object ROI features -> query initialization')
if hasattr(model_s2.transformer, 'roi_pool_layer'):
    print(f'roi_pool_layer: {model_s2.transformer.roi_pool_layer}')
    print(f'  Purpose: Pools 7x7 ROI features to 1x1')

print()
print('=== STAGE 2 FORWARD FLOW ===')
print('1. Backbone: image -> features [B, 256, H/32, W/32]')
print('2. ROI Align: extract features at GT box locations -> s_embed, o_embed')
print('3. so_linear(cat(s_embed, o_embed)) -> query initialization')
print('4. prepare_tag_query: pad to num_queries (zeros for unused)')
print('5. Transformer decoder -> rel_embed per SO pair')
print('6. memory_update: accumulate per so_track_id across frames')
print('7. At EOS: relation_classifier: mean_pool -> verb_class_embed -> pred_verb_logits')

Stage 2 model: VRDFormer_S2
Criterion: SetCriterion
Weight dict: {'loss_ce': 1, 'loss_ce_verb': 1, 'loss_bbox': 5, 'loss_giou': 2, 'loss_ce_0': 1, 'loss_ce_verb_0': 1, 'loss_bbox_0': 5, 'loss_giou_0': 2, 'loss_ce_1': 1, 'loss_ce_verb_1': 1, 'loss_bbox_1': 5, 'loss_giou_1': 2, 'loss_ce_2': 1, 'loss_ce_verb_2': 1, 'loss_bbox_2': 5, 'loss_giou_2': 2, 'loss_ce_3': 1, 'loss_ce_verb_3': 1, 'loss_bbox_3': 5, 'loss_giou_3': 2, 'loss_ce_4': 1, 'loss_ce_verb_4': 1, 'loss_bbox_4': 5, 'loss_giou_4': 2, 'loss_ce_enc': 1, 'loss_ce_verb_enc': 1, 'loss_bbox_enc': 5, 'loss_giou_enc': 2}

=== STAGE 2 SPECIFIC MODULES ===
so_linear: Linear(in_features=512, out_features=256, bias=True)
  Purpose: Fuses subject + object ROI features -> query initialization
roi_pool_layer: AvgPool2d(kernel_size=[7, 7], stride=[7, 7], padding=0)
  Purpose: Pools 7x7 ROI features to 1x1

=== STAGE 2 FORWARD FLOW ===
1. Backbone: image -> features [B, 256, H/32, W/32]
2. ROI Align: extract features at GT box locations -> s_emb

In [16]:
# Compare parameters between stages
n_s1 = count_params(model)
n_s2 = count_params(model_s2)

print('=== STAGE 1 vs STAGE 2 PARAMETERS ===')
print(f'Stage 1: {n_s1/1e6:.2f}M trainable params')
print(f'Stage 2: {n_s2/1e6:.2f}M trainable params')
print(f'Stage 2 extra: so_linear (d_model*2 -> d_model)')

=== STAGE 1 vs STAGE 2 PARAMETERS ===
Stage 1: 60.43M trainable params
Stage 2: 60.56M trainable params
Stage 2 extra: so_linear (d_model*2 -> d_model)


## 9. Understanding the Hungarian Matcher

The Hungarian matcher is used in Stage 1 to assign predictions to ground truth. It computes a cost matrix and finds the optimal bipartite matching.

In [17]:
print('=== HUNGARIAN MATCHER ===')
print()
print('Cost Matrix Components:')
print(f'  Subject class cost:  weight={args.set_cost_sub_class}')
print(f'  Object class cost:   weight={args.set_cost_obj_class}')
print(f'  Verb class cost:     weight={args.set_cost_verb_class} (multi-label cosine similarity)')
print(f'  Bbox L1 cost:        weight={args.set_cost_bbox} (max of sub/obj L1)')
print(f'  GIoU cost:           weight={args.set_cost_giou} (max of sub/obj GIoU)')
print()
print('Final cost matrix: (num_preds, num_targets)')
print('  C[i,j] = class_cost + bbox_cost + giou_cost')
print('  Hungarian algorithm: argmin_{assignment} sum_i C[i, assignment[i]]')
print()
print('Tracking-aware matching:')
print('  Track queries that match a previous target: cost[i, matched_j] = -1 (forced match)')
print('  Track queries that are false positives:    cost[i, :] = inf (cannot match any GT)')

=== HUNGARIAN MATCHER ===

Cost Matrix Components:
  Subject class cost:  weight=0.5
  Object class cost:   weight=0.5
  Verb class cost:     weight=1 (multi-label cosine similarity)
  Bbox L1 cost:        weight=5 (max of sub/obj L1)
  GIoU cost:           weight=2 (max of sub/obj GIoU)

Final cost matrix: (num_preds, num_targets)
  C[i,j] = class_cost + bbox_cost + giou_cost
  Hungarian algorithm: argmin_{assignment} sum_i C[i, assignment[i]]

Tracking-aware matching:
  Track queries that match a previous target: cost[i, matched_j] = -1 (forced match)
  Track queries that are false positives:    cost[i, :] = inf (cannot match any GT)


## 10. Final Tensor Flow Diagram (Stage 1, Full Forward Pass)

```
INPUT IMAGES                 (B, 3, H, W)
  |
  v  ResNet101 Backbone
FEATURES                     (B, 2048, H/32, W/32)  [+ pos: (B, 256, H/32, W/32)]
  |
  v  input_proj (1x1 Conv)
PROJECTED FEATURES           (B, 256, H/32, W/32)
  |
  v  Flatten + add level_embed
ENCODER INPUT                (H/32*W/32, B, 256)  = total_seq tokens
  |
  v  6x TransformerEncoderLayer (self-attention + FFN)
MEMORY                       (total_seq, B, 256)
  |
  v  Query Preparation:
  |    track_queries:  tgt = prev_hs_embed,  query_pos = ZEROS      (M queries)
  |    static_queries: tgt = ZEROS,          query_pos = LEARNED     (100 queries)
  |    concatenated:   tgt [M+100, B, 256],  query_pos [M+100, B, 256]
  |
  v  6x TransformerDecoderLayer (self-attn + cross-attn + FFN)
DECODER OUTPUT               (6, M+100, B, 256)  -> hs
  |
  v  Per-layer prediction heads:
  |    sub_class:  (B, M+100, 36)      subject object class logits
  |    obj_class:  (B, M+100, 36)      object object class logits
  |    verb_class: (B, M+100, 132)     verb logits (sigmoid, multi-label)
  |    sub_boxes:  (B, M+100, 4)       subject boxes (cx,cy,w,h, sigmoid)
  |    obj_boxes:  (B, M+100, 4)       object boxes
  |
  v  Output dict
FINAL OUTPUT:
  - pred_sub_logits:  (B, M+100, 36)
  - pred_obj_logits:  (B, M+100, 36)
  - pred_verb_logits: (B, M+100, 132)
  - pred_sub_boxes:   (B, M+100, 4)
  - pred_obj_boxes:   (B, M+100, 4)
  - hs_embed:         (B, M+100, 256)  -> becomes track_query_hs_embeds for next frame!
  - aux_outputs:      [5 dicts for intermediate layers]  (if aux_loss=True)
```

**Key insight:** `hs_embed` from the output is extracted and stored for matched queries.
It becomes `track_query_hs_embeds` in the next frame's target. This is the **recurrent mechanism**
that enables tracking without an explicit tracking module.

## 11. Summary: Answers to Key Questions

### What is the input shape to the model?

**Stage 1:** `(B, 3, H, W)` — batched images (NestedTensor), plus tracking info in targets.
Typical: `(4, 3, ~480, ~840)` after resize.

**Stage 2:** `(1, 3, H, W)` per frame, iterated over `seq_len=8` frames. Batch size is always 1.
Typical per frame: `(1, 3, ~480, ~840)`.

### How does the model understand tracking and relations?

**Tracking (Stage 1):** The `hs_embed` from the previous frame's decoder output is recycled
as content (tgt) for the current frame's track queries. Hungarian matching links predictions
to targets via track IDs. Track queries are force-matched to their known GT indices.

**Relations (Stage 2):** Ground-truth subject/object boxes initialize queries via ROI Align.
A memory dict keyed by `"sub_tid-obj_tid"` accumulates relation embeddings per tracklet.
At end-of-clip, `relation_classifier` mean-pools and classifies.

### Model Architecture Summary

| Component | Config |
|-----------|--------|
| Backbone | ResNet-101 (FrozenBatchNorm) |
| Feature channels | 2048 → projected to 256 |
| Encoder layers | 6 |
| Decoder layers | 6 |
| Hidden dimension | 256 |
| Attention heads | 8 |
| FFN dimension | 2048 |
| Dropout | 0.1 |
| Object queries | 100 (+ M tracking queries) |
| Prediction heads | 5 per layer: sub_cls, obj_cls, verb_cls, sub_box, obj_box |

**Next:** Notebook 4 will run the full training loop with loss computation and evaluation.